# Estructura local: g(r), ángulos de enlace y anillos

Analiza **un frame** (`MODO = "frame"`) o **la media de una dinámica** (`MODO = "media"`) de una lámina 2D (BGDY u otra) o de un sistema 3D. Lee cualquier formato de ASE: extxyz, output de pw.x (`FORMATO = "espresso-out"`), .traj, etc.

Cambia los parámetros de la celda siguiente y ejecuta todo (*Run → Run All Cells*). Las tablas, figuras y archivos se guardan en `OUT`.

**Convenios**
- **g(r)**: distancias 3D con todas las imágenes periódicas. En una lámina (pbc z = F o más de 5 Å de vacío en c) se normaliza por el área de la celda con coronas 2πr dr, así que g(r) → 1 a r grande. No se cuentan imágenes a través del vacío. Para las parciales, g_ab = g_ba. *n_X_Y(r)* es el número medio de átomos Y a menos de r de un X; no depende de la normalización.
- **Enlaces**: d < corte del par (`RC`, y `RC_PAR` para pares concretos). De ellos salen la coordinación, los ángulos y los anillos.
- **Ángulos**: triplete X-Y-Z con Y central; P(θ) normalizada a 1. `ANGULOS_SIN = True` dibuja P(θ)/sin θ: sin esa corrección, una cadena lineal que vibra da el máximo hacia 170° y no en 180°.
- **Anillos**: primitivos o de camino más corto (sin atajos; Franzblau 1991, Yuan y Cormack 2002), buscados en la red periódica infinita. Así no se cuentan los ciclos que dan la vuelta a la celda. La BGDY perfecta solo tiene anillos de 30; cuando se rompe una cadena, dos se unen en uno de 50.

In [ ]:
# ---------------- Parámetros ----------------
ARCHIVO = "a2000.xyz"   # extxyz, output de pw.x, .traj, ...
FORMATO = None                   # None = lo deduce ASE; "espresso-out" para un output de pw.x

MODO  = "frame"    # "frame": una configuración  |  "media": media sobre la dinámica
FRAME = -1       # modo "frame": índice del frame (0 = primero, -1 = último)

# solo en modo "media"
DESCARTE_FS    = 0.0   # fs iniciales que no entran (0 = toda la dinámica; el dataset AIMD descarta 350)
STRIDE         = 1     # 1 de cada STRIDE frames para g(r), ángulos y anillos
STRIDE_ANILLOS = 1     # de esos, 1 de cada N solo para los anillos (lo más lento, ~0.05 s/frame en la 3x3)

# g(r)
DIM   = "auto"   # "auto" | 2 (lámina: normaliza por área) | 3 (volumen)
RMAX  = 20.0     # Å
DR    = 0.02     # Å
SIGMA = "auto"   # Å, ensanchamiento gaussiano; 0 = histograma tal cual.
                 # "auto" = 0.03 Å en modo "frame" (126 átomos dan un histograma muy ruidoso) y 0 en "media"

# enlaces (coordinación, ángulos y anillos)
RC     = 1.85           # Å, corte para todos los pares...
RC_PAR = {"B-C": 2.4}   # ...salvo estos. A 2000 K el B-C llega a 2.0-2.2 Å sin romperse: el hueco
                        # de g_BC está entre 2.2 y 2.4 Å, y 2.2 es también el criterio de rotura B-C.
                        # Con {} se usa 1.85 para todo. Un corte 0 desactiva ese par (p. ej. "B-B": 0).
DTHETA      = 1.0       # grados
ANGULOS_SIN = False     # True: dibuja P(θ)/sin θ
MAX_ANILLO  = 60        # tamaño máximo de anillo buscado

OUT = None              # carpeta de salida; None = estructura_<archivo>_<frameN | media>

In [ ]:
# Funciones de estructura_local.py (idénticas al script).
# Celda plegada: haz clic en los puntos suspensivos para verla.

import argparse
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from ase import Atoms
from ase.io import read, write
from ase.neighborlist import neighbor_list

RC_DEF = 1.85          # Å, corte de enlace por defecto (todos los pares)
VACIO_MIN = 5.0        # Å de vacío a lo largo de c para tratar el sistema como lámina


# ----------------------------------------------------------------------------
# Utilidades
# ----------------------------------------------------------------------------
def _lista(frames):
    if isinstance(frames, Atoms):
        return [frames]
    return list(frames)


def detectar_dim(atoms: Atoms, vacio_min: float = VACIO_MIN) -> int:
    """2 si no hay periodicidad en z o si hay más de `vacio_min` Å de vacío en c."""
    if not atoms.pbc[2]:
        return 2
    cell = np.asarray(atoms.cell)
    nrm = np.cross(cell[0], cell[1])
    nrm /= np.linalg.norm(nrm)
    h = abs(np.dot(cell[2], nrm))
    s = np.sort(atoms.get_scaled_positions(wrap=True)[:, 2])
    huecos = np.diff(np.concatenate([s, [s[0] + 1.0]]))
    return 2 if huecos.max() * h > vacio_min else 3


def preparar(atoms: Atoms, dim="auto"):
    """Copia envuelta en la celda; en 2D, sin periodicidad en z."""
    a = atoms.copy()
    a.calc = None
    if a.cell.rank < 2:
        raise ValueError("La configuración necesita al menos los vectores a y b de la celda.")
    d = detectar_dim(a) if str(dim) == "auto" else int(dim)
    if d == 2:
        a.pbc = [bool(a.pbc[0]), bool(a.pbc[1]), False]
    if a.cell.rank < 3:
        a.set_cell(a.cell.complete(), scale_atoms=False)
    a.wrap()
    return a, d


def medida(a: Atoms, dim: int) -> float:
    cell = np.asarray(a.cell)
    if dim == 2:
        return float(np.linalg.norm(np.cross(cell[0], cell[1])))
    return float(abs(np.linalg.det(cell)))


def _suavizar(y, s_bins):
    if s_bins <= 0:
        return y
    m = int(np.ceil(4 * s_bins))
    x = np.arange(-m, m + 1)
    k = np.exp(-0.5 * (x / s_bins) ** 2)
    k /= k.sum()
    return np.convolve(np.pad(y, m, mode="edge"), k, mode="valid")


def _tabla_cortes(rc_par):
    """Acepta {"B-C": 1.95}, {("B", "C"): 1.95} o ["B-C=1.95", ...]."""
    tabla = {}
    if not rc_par:
        return tabla
    if isinstance(rc_par, (list, tuple)):
        items = []
        for s in rc_par:
            par, v = s.split("=")
            items.append((par, float(v)))
    else:
        items = list(rc_par.items())
    for par, v in items:
        A, B = par.split("-") if isinstance(par, str) else par
        tabla[tuple(sorted((A.strip(), B.strip())))] = float(v)
    return tabla


def enlaces(a: Atoms, rc=RC_DEF, rc_par=None):
    """Pares enlazados (i, j, d, D, S) con cortes por par de elementos.

    D = r_j - r_i + S·celda (vector de enlace); S = imagen de j.
    Un corte de 0 para un par desactiva esos enlaces.
    """
    tabla = _tabla_cortes(rc_par)
    rmax = max([rc] + list(tabla.values()))
    i, j, d, D, S = neighbor_list("ijdDS", a, rmax)
    sym = np.array(a.get_chemical_symbols())
    corte = np.full(len(i), float(rc))
    for (A, B), v in tabla.items():
        m = ((sym[i] == A) & (sym[j] == B)) | ((sym[i] == B) & (sym[j] == A))
        corte[m] = v
    k = d < corte
    return i[k], j[k], d[k], D[k], S[k]


# ----------------------------------------------------------------------------
# g(r)
# ----------------------------------------------------------------------------
def gdr(frames, rmax=10.0, dr=0.02, dim="auto", sigma=0.0):
    """g(r) total y parciales, y n_a_b(r), promediados sobre los frames.

    sigma: ensanchamiento gaussiano (Å) de la distribución de distancias;
    0 = histograma. n(r) se calcula siempre sin ensanchar.
    """
    frames = _lista(frames)
    nb = int(round(rmax / dr))
    bordes = np.linspace(0.0, nb * dr, nb + 1)
    r = 0.5 * (bordes[1:] + bordes[:-1])
    sg, sn, cg, cn = defaultdict(lambda: np.zeros(nb)), defaultdict(lambda: np.zeros(nb)), Counter(), Counter()
    dims = set()
    for at in frames:
        a, d = preparar(at, dim)
        dims.add(d)
        M = medida(a, d)
        corona = np.pi * np.diff(bordes ** 2) if d == 2 else 4.0 / 3.0 * np.pi * np.diff(bordes ** 3)
        sym = np.array(a.get_chemical_symbols())
        N = len(a)
        i, j, dist = neighbor_list("ijd", a, bordes[-1])
        H = np.histogram(dist, bordes)[0].astype(float)
        if N > 1:
            sg["total"] += _suavizar(H, sigma / dr) / (N * (N - 1) / M * corona)
            cg["total"] += 1
        sn["total"] += np.cumsum(H) / N
        cn["total"] += 1
        si, sj = sym[i], sym[j]
        esp = sorted(set(sym))
        for A in esp:
            Na = int((sym == A).sum())
            for B in esp:
                Nb = int((sym == B).sum())
                Hab = np.histogram(dist[(si == A) & (sj == B)], bordes)[0].astype(float)
                sn[f"{A}_{B}"] += np.cumsum(Hab) / Na
                cn[f"{A}_{B}"] += 1
                if A <= B:
                    npar = Na * (Nb - (A == B))
                    if npar > 0:
                        sg[f"{A}-{B}"] += _suavizar(Hab, sigma / dr) / (npar / M * corona)
                        cg[f"{A}-{B}"] += 1
    g = {k: v / cg[k] for k, v in sg.items()}
    n = {k: v / cn[k] for k, v in sn.items()}
    return {"r": r, "dr": dr, "g": g, "n": n, "dim": sorted(dims), "nframes": len(frames)}


def _primer_pico(r, g, rlim=3.0, s_bins=2.5):
    gs = _suavizar(g, s_bins)
    m = r < rlim
    if not np.any(gs[m] > 0):
        return None
    k = int(np.argmax(gs[m] > 0.2 * gs[m].max()))
    while k + 1 < m.sum() and gs[k + 1] >= gs[k]:
        k += 1
    return float(r[k])


# ----------------------------------------------------------------------------
# Ángulos de enlace y coordinación
# ----------------------------------------------------------------------------
def angulos(frames, rc=RC_DEF, rc_par=None, dtheta=1.0, dim="auto"):
    """Distribución de ángulos de enlace por triplete X-Y-Z (Y central)."""
    frames = _lista(frames)
    nb = int(round(180.0 / dtheta))
    bordes = np.linspace(0.0, 180.0, nb + 1)
    theta = 0.5 * (bordes[1:] + bordes[:-1])
    H = defaultdict(lambda: np.zeros(nb))
    ntrip, suma = Counter(), Counter()
    coord = defaultdict(Counter)
    for at in frames:
        a, _ = preparar(at, dim)
        N = len(a)
        sym = np.array(a.get_chemical_symbols())
        i, j, _, D, _ = enlaces(a, rc, rc_par)
        o = np.argsort(i, kind="stable")
        i, j, D = i[o], j[o], D[o]
        z = np.bincount(i, minlength=N)
        for c in range(N):
            coord[sym[c]][int(z[c])] += 1
        ini = np.concatenate([[0], np.cumsum(z)])
        claves, ths = [], []
        for c in np.nonzero(z >= 2)[0]:
            v = D[ini[c]:ini[c + 1]]
            u = v / np.linalg.norm(v, axis=1)[:, None]
            p, q = np.triu_indices(len(v), 1)
            th = np.degrees(np.arccos(np.clip(np.einsum("ij,ij->i", u[p], u[q]), -1.0, 1.0)))
            vj = sym[j[ini[c]:ini[c + 1]]]
            x, y = vj[p], vj[q]
            lo, hi = np.where(x <= y, x, y), np.where(x <= y, y, x)
            claves.extend(f"{l}-{sym[c]}-{h}" for l, h in zip(lo, hi))
            ths.append(th)
        if not ths:
            continue
        claves = np.array(claves)
        ths = np.concatenate(ths)
        for k in ["total"] + sorted(set(claves)):
            t = ths if k == "total" else ths[claves == k]
            H[k] += np.histogram(t, bordes)[0]
            ntrip[k] += len(t)
            suma[k] += float(t.sum())
    P = {k: v / (v.sum() * dtheta) for k, v in H.items() if v.sum() > 0}
    # P(θ)/sin θ renormalizada: quita el factor de ángulo sólido, que lleva a 0 la
    # densidad en 180° aunque las cadenas sean lineales en promedio
    seno = np.sin(np.radians(theta))
    P_sin = {k: (v / seno) / ((v / seno).sum() * dtheta) for k, v in P.items()}
    media = {k: suma[k] / ntrip[k] for k in ntrip}
    return {"theta": theta, "dtheta": dtheta, "P": P, "P_sin": P_sin, "n": dict(ntrip), "media": media,
            "coord": {k: dict(sorted(v.items())) for k, v in coord.items()}, "nframes": len(frames)}


# ----------------------------------------------------------------------------
# Anillos primitivos en la red periódica infinita
# ----------------------------------------------------------------------------
def _bfs(raiz, vec, prof):
    """BFS desde (raiz, imagen 0) hasta profundidad `prof`: distancias y padres."""
    ini = (raiz, 0, 0, 0)
    dist = {ini: 0}
    padres = {ini: []}
    frontera = [ini]
    for k in range(1, prof + 1):
        nueva = []
        for st in frontera:
            a0, x, y, z = st
            for b, sx, sy, sz in vec[a0]:
                nbst = (b, x + sx, y + sy, z + sz)
                dk = dist.get(nbst)
                if dk is None:
                    dist[nbst] = k
                    padres[nbst] = [st]
                    nueva.append(nbst)
                elif dk == k:
                    padres[nbst].append(st)
        frontera = nueva
        if not frontera:
            break
    return dist, padres


def _caminos(t, padres, raiz, cache, cap, aviso):
    """Todos los caminos más cortos raiz -> t que solo pasan por átomos >= raiz."""
    r = cache.get(t)
    if r is not None:
        return r
    if t[0] < raiz:
        cache[t] = []
        return cache[t]
    ps = padres[t]
    if not ps:
        r = [(t,)]
    else:
        r = [c + (t,) for p in ps for c in _caminos(p, padres, raiz, cache, cap, aviso)]
        if len(r) > cap:
            r = r[:cap]
            aviso[0] = True
    cache[t] = r
    return r


def _canonica(ring):
    """Forma independiente del origen, del sentido y de la imagen de partida."""
    n = len(ring)
    amin = min(s[0] for s in ring)
    mejor = None
    for p, s in enumerate(ring):
        if s[0] != amin:
            continue
        _, x0, y0, z0 = s
        for sentido in (1, -1):
            seq = tuple((ring[(p + sentido * m) % n][0],
                         ring[(p + sentido * m) % n][1] - x0,
                         ring[(p + sentido * m) % n][2] - y0,
                         ring[(p + sentido * m) % n][3] - z0) for m in range(n))
            if mejor is None or seq < mejor:
                mejor = seq
    return mejor


def _sin_atajos(ring, dist):
    n = len(ring)
    for p in range(n):
        a, x, y, z = ring[p]
        da = dist[a]
        for q in range(p + 2, n):
            dr = min(q - p, n - (q - p))
            if dr <= 1:
                continue
            b, bx, by, bz = ring[q]
            dg = da.get((b, bx - x, by - y, bz - z))
            if dg is not None and dg < dr:
                return False
    return True


def _composicion(simbolos):
    c = Counter(simbolos)
    return "".join(f"{e}{c[e]}" for e in sorted(c))


def anillos(atoms: Atoms, rc=RC_DEF, rc_par=None, max_anillo=60, dim="auto", max_caminos=20000):
    """Anillos primitivos (sin atajos) de hasta `max_anillo` átomos de una configuración.

    Devuelve un dict con:
      anillos     lista de dicts (n, atomos, composicion), ordenada por tamaño
      tamanos     Counter tamaño -> número de anillos en la celda
      frac_atomos dict tamaño -> fracción de átomos en al menos un anillo de ese tamaño
      anillo_min  por átomo, tamaño del menor anillo al que pertenece (0 = ninguno)
      n_anillos   por átomo, número de anillos a los que pertenece
      coord       por átomo, número de vecinos enlazados
    """
    if max_anillo < 3:
        raise ValueError("max_anillo debe ser >= 3")
    t0 = time.time()
    a, _ = preparar(atoms, dim)
    N = len(a)
    sym = np.array(a.get_chemical_symbols())
    i, j, _, _, S = enlaces(a, rc, rc_par)
    vec = [[] for _ in range(N)]
    for ii, jj, s in zip(i.tolist(), j.tolist(), S.astype(int).tolist()):
        vec[ii].append((jj, s[0], s[1], s[2]))
    coord = np.array([len(v) for v in vec])
    prof = max_anillo // 2
    dist = [None] * N
    cand = {}
    aviso = [False]
    for raiz in range(N):
        dd, padres = _bfs(raiz, vec, prof)
        dist[raiz] = dd
        if coord[raiz] < 2:
            continue
        cache = {}
        niveles = defaultdict(list)
        for st, k in dd.items():
            if k >= 1 and st[0] >= raiz:
                niveles[k].append(st)
        for k in sorted(niveles):
            for t in niveles[k]:
                # anillo par de 2k: dos caminos más cortos disjuntos hasta t
                if k >= 2 and 2 * k <= max_anillo and len(padres[t]) >= 2:
                    P = _caminos(t, padres, raiz, cache, max_caminos, aviso)
                    for x in range(len(P)):
                        p = P[x]
                        for y in range(x + 1, len(P)):
                            q = P[y]
                            if p[1] == q[1] or p[-2] == q[-2]:
                                continue
                            if set(p[1:-1]).isdisjoint(q[1:-1]):
                                ring = list(p) + list(q[-2:0:-1])
                                cand.setdefault(_canonica(ring), ring)
                # anillo impar de 2k+1: caminos hasta t y hasta u, vecinos a la misma distancia
                if 2 * k + 1 <= max_anillo:
                    a0, x0, y0, z0 = t
                    for b, sx, sy, sz in vec[a0]:
                        u = (b, x0 + sx, y0 + sy, z0 + sz)
                        if b < raiz or u <= t or dd.get(u) != k:
                            continue
                        Pt = _caminos(t, padres, raiz, cache, max_caminos, aviso)
                        Pu = _caminos(u, padres, raiz, cache, max_caminos, aviso)
                        for p in Pt:
                            for q in Pu:
                                if p[1] == q[1]:
                                    continue
                                if set(p[1:]).isdisjoint(q[1:]):
                                    ring = list(p) + list(q[:0:-1])
                                    cand.setdefault(_canonica(ring), ring)
    lista = []
    anillo_min = np.zeros(N, dtype=int)
    n_anillos = np.zeros(N, dtype=int)
    por_tam = defaultdict(set)
    for ring in cand.values():
        if not _sin_atajos(ring, dist):
            continue
        at = [s[0] for s in ring]
        n = len(ring)
        lista.append({"n": n, "atomos": at, "composicion": _composicion(sym[at])})
        for x in set(at):
            n_anillos[x] += 1
            anillo_min[x] = n if anillo_min[x] == 0 else min(anillo_min[x], n)
            por_tam[n].add(x)
    lista.sort(key=lambda d: (d["n"], d["atomos"]))
    if aviso[0]:
        print(f"  aviso: más de {max_caminos} caminos más cortos hasta algún átomo; "
              "el grafo está muy conectado (¿corte de enlace demasiado grande?) y "
              "puede faltar algún anillo.", file=sys.stderr)
    return {"anillos": lista, "tamanos": Counter(d["n"] for d in lista),
            "frac_atomos": {n: len(s) / N for n, s in por_tam.items()},
            "anillo_min": anillo_min, "n_anillos": n_anillos, "coord": coord,
            "N": N, "max_anillo": max_anillo, "atoms": a, "segundos": time.time() - t0}


def resumen_anillos(resultados):
    """Media sobre frames: anillos por celda, por átomo y fracción de átomos."""
    resultados = [resultados] if isinstance(resultados, dict) else list(resultados)
    tam = sorted(set().union(*[r["tamanos"].keys() for r in resultados]))
    filas = []
    for n in tam:
        cnt = np.array([r["tamanos"].get(n, 0) for r in resultados], float)
        por_at = np.array([r["tamanos"].get(n, 0) / r["N"] for r in resultados])
        frac = np.array([r["frac_atomos"].get(n, 0.0) for r in resultados])
        filas.append((n, cnt.mean(), cnt.std(), por_at.mean(), frac.mean()))
    return filas


# ----------------------------------------------------------------------------
# Figuras (cada una en sus propios ejes, para el notebook; `figura` las junta)
# ----------------------------------------------------------------------------
_TINTA, _TINTA2, _REJILLA = "#0b0b0b", "#52514e", "#e4e3df"
_SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]


def _estilo(ax, titulo, xl, yl):
    ax.set_title(titulo, loc="left", fontsize=11, color=_TINTA)
    ax.set_xlabel(xl, color=_TINTA2)
    ax.set_ylabel(yl, color=_TINTA2)
    ax.grid(True, color=_REJILLA, lw=0.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#a9a8a3")
    ax.tick_params(colors=_TINTA2, labelsize=9)


def _ejes(ax, figsize=(6.4, 4.2)):
    import matplotlib.pyplot as plt
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        return fig, ax
    return ax.figure, ax


def _norma(G):
    return "lámina, normalizada por área" if G["dim"] == [2] else "normalizada por volumen"


def plot_gr(G, ax=None, titulo=None):
    """g(r) total y parciales en la misma gráfica."""
    fig, ax = _ejes(ax)
    r = G["r"]
    for c, k in zip(_SERIES, sorted(k for k in G["g"] if k != "total")):
        ax.plot(r, G["g"][k], color=c, lw=1.5, label=k)
    if "total" in G["g"]:
        ax.plot(r, G["g"]["total"], color=_TINTA, lw=1.0, label="total")
    ax.axhline(1.0, color=_TINTA2, lw=0.6, ls=":")
    _estilo(ax, titulo or f"g(r) total y parciales  ({_norma(G)})", "r (Å)", "g(r)")
    ax.set_xlim(0, r[-1])
    ax.legend(frameon=False, fontsize=9)
    return fig


def plot_gr_total(G, ax=None, titulo=None):
    """Solo la g(r) total."""
    fig, ax = _ejes(ax)
    r = G["r"]
    ax.plot(r, G["g"]["total"], color=_TINTA, lw=1.5)
    ax.axhline(1.0, color=_TINTA2, lw=0.6, ls=":")
    _estilo(ax, titulo or f"g(r) total  ({_norma(G)})", "r (Å)", "g(r)")
    ax.set_xlim(0, r[-1])
    ax.set_ylim(bottom=0)
    return fig


def plot_angulos(A, ax=None, angulos_sin=False, max_tripletes=4, titulo=None):
    """P(θ) de los tripletes más frecuentes (hasta `max_tripletes`)."""
    fig, ax = _ejes(ax)
    tr = sorted((k for k in A["P"] if k != "total"), key=lambda k: -A["n"][k])[:max_tripletes]
    P = A["P_sin"] if angulos_sin else A["P"]
    for c, k in zip(_SERIES, sorted(tr)):
        ax.plot(A["theta"], P[k], color=c, lw=1.5, label=f"{k}  ({A['n'][k]})")
    _estilo(ax, titulo or ("Ángulos de enlace" + ("  (P/sin θ)" if angulos_sin else "")), "θ (grados)",
            "P(θ)/sin θ (renormalizada)" if angulos_sin else "P(θ) (grados⁻¹)")
    ax.set_xlim(0, 180)
    ax.set_xticks(range(0, 181, 30))
    ax.legend(frameon=False, fontsize=9, title="X-Y-Z, Y central (n.º de ángulos)", title_fontsize=8)
    return fig


def plot_anillos(R, ax=None, titulo=None):
    """Anillos por celda según su tamaño (media si R es una lista de frames)."""
    fig, ax = _ejes(ax)
    filas = resumen_anillos(R)
    if filas:
        n = [f[0] for f in filas]
        m = [f[1] for f in filas]
        x = np.arange(len(n))          # tamaños como categorías: barras igual de anchas
        ax.bar(x, m, width=0.7, color=_SERIES[0], edgecolor="white", linewidth=1.0)
        for xi, y in zip(x, m):
            ax.text(xi, y, f"{y:g}" if y == int(y) else (f"{y:.2f}" if y >= 1 else f"{y:.2g}"),
                    ha="center", va="bottom", fontsize=8, color=_TINTA2)
        ax.set_xticks(x, [str(k) for k in n])
        ax.set_xlim(-0.6, max(len(n), 3) - 0.4)
        ax.tick_params(axis="x", labelrotation=90 if len(n) > 12 else 0)
    nf = len(R) if isinstance(R, list) else 1
    _estilo(ax, titulo or ("Anillos primitivos" + (f" (media de {nf} frames)" if nf > 1 else "")),
            "átomos en el anillo", "anillos por celda")
    return fig


def plot_cortes(G, rc=RC_DEF, rc_par=None, rlim=(1.0, 3.0), ax=None):
    """g parciales en escala log entre rlim, con el corte de enlace de cada par.

    El corte debe caer en el hueco entre primeros y segundos vecinos.
    """
    fig, ax = _ejes(ax)
    tabla = _tabla_cortes(rc_par)
    r = G["r"]
    m = (r >= rlim[0]) & (r <= rlim[1])
    for c, k in zip(_SERIES, sorted(k for k in G["g"] if k != "total")):
        y = np.where(G["g"][k][m] > 0, G["g"][k][m], np.nan)
        if np.all(np.isnan(y)):
            continue
        ax.plot(r[m], y, color=c, lw=1.5, label=k)
        rxy = tabla.get(tuple(sorted(k.split("-"))), rc)
        if rxy > 0:
            ax.axvline(rxy, color=c, lw=1.0, ls="--")
            ax.text(rxy, 1.0, f" {rxy:g}", color=_TINTA2, fontsize=8, transform=ax.get_xaxis_transform(),
                    ha="left", va="top")
    ax.set_yscale("log")
    _estilo(ax, "Corte de enlace por par (línea discontinua)", "r (Å)", "g(r) (escala log)")
    ax.set_xlim(*rlim)
    ax.legend(frameon=False, fontsize=9, loc="best")
    return fig


def figura(G=None, A=None, R=None, archivo="estructura.png", angulos_sin=False):
    """Las tres gráficas en una fila, guardadas en `archivo`."""
    import matplotlib.pyplot as plt
    paneles = [(f, x) for f, x in ((plot_gr, G), (plot_angulos, A), (plot_anillos, R)) if x]
    fig, axs = plt.subplots(1, len(paneles), figsize=(5.2 * len(paneles), 4.0), squeeze=False)
    for ax, (f, x) in zip(axs[0], paneles):
        if f is plot_angulos:
            f(x, ax=ax, angulos_sin=angulos_sin)
        else:
            f(x, ax=ax)
    fig.tight_layout()
    fig.savefig(archivo, dpi=160)
    plt.close(fig)
    return archivo


# ----------------------------------------------------------------------------
# Informes y archivos (los usan el programa y el notebook)
# ----------------------------------------------------------------------------
def _guardar_csv(ruta, cabecera, columnas):
    np.savetxt(ruta, np.column_stack(columnas), delimiter=",", header=",".join(cabecera),
               comments="", fmt="%.6g")


def anillos_frames(frames, rc=RC_DEF, rc_par=None, max_anillo=60, dim="auto", cada=1, progreso=True):
    """anillos() sobre 1 de cada `cada` frames; guarda step y time_fs si el frame los trae."""
    sel = _lista(frames)[::max(1, int(cada))]
    R, t0 = [], time.time()
    for k, at in enumerate(sel):
        res = anillos(at, rc, rc_par, max_anillo, dim)
        res["info"] = {c: at.info[c] for c in ("step", "time_fs") if c in at.info}
        R.append(res)
        if progreso and len(sel) > 1 and ((k + 1) % max(1, len(sel) // 10) == 0 or k + 1 == len(sel)):
            print(f"  anillos: {k + 1}/{len(sel)} frames ({time.time() - t0:.0f} s)", flush=True)
    return R


def tabla_gr(G, rc=RC_DEF, rc_par=None, delta=0.15):
    """Por par X-Y: primer pico de g, corte usado y n_X_Y en rc - delta, rc y rc + delta.

    Si n cambia entre rc - delta y rc + delta, hay distancias cerca del corte.
    """
    tabla = _tabla_cortes(rc_par)
    filas = []
    for k in sorted(c for c in G["n"] if c != "total"):
        X, Y = k.split("_")
        par = tuple(sorted((X, Y)))
        rxy = tabla.get(par, rc)
        gk = G["g"].get("-".join(par))
        pico = _primer_pico(G["r"], gk) if gk is not None else None
        n = lambda x: float(np.interp(x, G["r"] + G["dr"] / 2, G["n"][k]))
        filas.append({"par": f"{X}-{Y}", "primer_pico_A": pico, "corte_A": rxy,
                      "n(corte-d)": n(rxy - delta), "n(corte)": n(rxy), "n(corte+d)": n(rxy + delta)})
    return filas


def tabla_coordinacion(A):
    zs = sorted({z for c in A["coord"].values() for z in c})
    filas = []
    for e, c in sorted(A["coord"].items()):
        tot = sum(c.values())
        filas.append({"elemento": e, **{f"z={z}": c.get(z, 0) / tot for z in zs}})
    return filas


def tabla_angulos(A):
    return [{"triplete": k, "angulos_por_frame": A["n"][k] / A["nframes"], "theta_medio": A["media"][k]}
            for k in ["total"] + sorted(k for k in A["P"] if k != "total")]


def tabla_anillos(R):
    return [{"n": n, "por_celda": m, "desv": s, "por_atomo": pa, "frac_atomos": fr}
            for n, m, s, pa, fr in resumen_anillos(R)]


def _imprimir(filas, fmt="{:.4g}"):
    if not filas:
        print("  (nada)")
        return
    cols = list(filas[0])
    txt = [[c for c in cols]] + [["-" if f[c] is None else (fmt.format(f[c]) if isinstance(f[c], float) else str(f[c]))
                                  for c in cols] for f in filas]
    w = [max(len(t[i]) for t in txt) for i in range(len(cols))]
    for t in txt:
        print("  " + "  ".join(x.rjust(wi) for x, wi in zip(t, w)))


def guardar_gr(G, out):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    cg = ["total"] + sorted(k for k in G["g"] if k != "total")
    cn = ["total"] + sorted(k for k in G["n"] if k != "total")
    _guardar_csv(out / "gr.csv", ["r"] + [f"g_{k}" for k in cg] + [f"n_{k}" for k in cn],
                 [G["r"]] + [G["g"][k] for k in cg] + [G["n"][k] for k in cn])


def guardar_angulos(A, out):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    claves = ["total"] + sorted(k for k in A["P"] if k != "total")
    _guardar_csv(out / "angulos.csv", ["theta"] + [f"P_{k}" for k in claves] + [f"Psin_{k}" for k in claves],
                 [A["theta"]] + [A["P"][k] for k in claves] + [A["P_sin"][k] for k in claves])
    filas = tabla_coordinacion(A)
    with open(out / "coordinacion.csv", "w") as f:
        f.write(",".join(filas[0]) + "\n")
        for fi in filas:
            f.write(",".join(str(v) if isinstance(v, str) else f"{v:.6g}" for v in fi.values()) + "\n")


def guardar_anillos(R, out):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "anillos.csv", "w") as f:
        f.write("n,anillos_por_celda,desv,anillos_por_atomo,frac_atomos\n")
        for n, m, s, pa, fr in resumen_anillos(R):
            f.write(f"{n},{m:.6g},{s:.6g},{pa:.6g},{fr:.6g}\n")
    with open(out / "anillos_lista.csv", "w") as f:
        f.write("frame,step,time_fs,n,composicion,atomos\n")
        for k, res in enumerate(R):
            st, tf = res.get("info", {}).get("step", ""), res.get("info", {}).get("time_fs", "")
            for d in res["anillos"]:
                f.write(f"{k},{st},{tf},{d['n']},{d['composicion']},{' '.join(map(str, d['atomos']))}\n")
    ult = R[-1]["atoms"].copy()
    ult.arrays["coord"] = R[-1]["coord"]
    ult.arrays["anillo_min"] = R[-1]["anillo_min"]
    ult.arrays["n_anillos"] = R[-1]["n_anillos"]
    write(out / "atomos_anillos.extxyz", ult, format="extxyz")


def mostrar(filas, decimales=4):
    """Tabla como DataFrame de pandas si está instalado; si no, en texto."""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(filas).round(decimales))
    except ImportError:
        _imprimir(filas)


## 1. Lectura

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if MODO == "frame":
    frames = [read(ARCHIVO, index=FRAME, format=FORMATO)]
    etiqueta = f"frame{FRAME}"
elif MODO == "media":
    frames = _lista(read(ARCHIVO, index=":", format=FORMATO))
    if DESCARTE_FS > 0:
        t = np.array([a.info.get("time_fs", np.nan) for a in frames], float)
        if not np.all(np.isfinite(t)):
            raise ValueError("Los frames no traen time_fs: pon DESCARTE_FS = 0 y recorta con STRIDE o un slice.")
        frames = [a for a, ti in zip(frames, t) if ti >= DESCARTE_FS]
    frames = frames[::STRIDE]
    etiqueta = "media"
else:
    raise ValueError('MODO debe ser "frame" o "media"')

OUT = Path(OUT or f"estructura_{Path(ARCHIVO).stem}_{etiqueta}")
OUT.mkdir(parents=True, exist_ok=True)

a0, d0 = preparar(frames[0], DIM)
print(f"{ARCHIVO}: {len(frames)} frame(s), {len(a0)} átomos ({a0.get_chemical_formula()}), "
      f"{'lámina 2D' if d0 == 2 else '3D'}, {'área' if d0 == 2 else 'volumen'} {medida(a0, d0):.2f} "
      f"Å{'²' if d0 == 2 else '³'}")
t = np.array([a.info.get("time_fs", np.nan) for a in frames], float)
T = np.array([a.info.get("T_inst_K", np.nan) for a in frames], float)
if np.isfinite(t).any():
    print(f"tiempo: {np.nanmin(t):.1f} - {np.nanmax(t):.1f} fs" + (f"   (step {frames[0].info.get('step')})" if len(frames) == 1 else ""))
if np.isfinite(T).any():
    print(f"T_inst: {np.nanmean(T):.0f} K" + (f" (desv. {np.nanstd(T):.0f} K)" if len(frames) > 1 else ""))
print(f"cortes de enlace: {RC} Å" + "".join(f", {x}-{y} {v} Å" for (x, y), v in _tabla_cortes(RC_PAR).items()))
print(f"salida en {OUT}/")

## 2. g(r)

Primero la g(r) total con las parciales, tal cual; después, solo la total.

In [ ]:
sigma = (0.03 if MODO == "frame" else 0.0) if SIGMA == "auto" else float(SIGMA)
G = gdr(frames, rmax=RMAX, dr=DR, dim=DIM, sigma=sigma)
print(f"ensanchamiento gaussiano: {sigma} Å")
guardar_gr(G, OUT)
titulo = f"frame {FRAME}" if MODO == "frame" else f"media de {len(frames)} frames"

fig = plot_gr(G, titulo=f"g(r) total y parciales, {titulo}")
fig.tight_layout(); fig.savefig(OUT / "gr.png", dpi=160); plt.show()

fig = plot_gr_total(G, titulo=f"g(r) total, {titulo}")
fig.tight_layout(); fig.savefig(OUT / "gr_total.png", dpi=160); plt.show()

### Comprobación de los cortes de enlace

El corte de cada par (línea discontinua) debe caer en el hueco entre primeros y segundos vecinos. En la tabla, si *n* cambia entre `corte-d` y `corte+d` (d = 0.15 Å), hay distancias cerca del corte. En ese caso los anillos y la coordinación dependen del corte elegido.

In [ ]:
fig = plot_cortes(G, RC, RC_PAR)
fig.tight_layout(); fig.savefig(OUT / "gr_cortes.png", dpi=160); plt.show()
mostrar(tabla_gr(G, RC, RC_PAR), 3)

## 3. Ángulos de enlace y coordinación

In [ ]:
A = angulos(frames, rc=RC, rc_par=RC_PAR, dtheta=DTHETA, dim=DIM)
guardar_angulos(A, OUT)

fig = plot_angulos(A, angulos_sin=ANGULOS_SIN)
fig.tight_layout(); fig.savefig(OUT / "angulos.png", dpi=160); plt.show()

print("Coordinación (fracción de átomos con z vecinos enlazados)")
mostrar(tabla_coordinacion(A), 4)
print("Ángulos por triplete X-Y-Z (Y central)")
mostrar(tabla_angulos(A), 1)

## 4. Anillos

In [ ]:
R = anillos_frames(frames, rc=RC, rc_par=RC_PAR, max_anillo=MAX_ANILLO, dim=DIM, cada=STRIDE_ANILLOS)
guardar_anillos(R, OUT)
print(f"{len(R)} frame(s), {sum(r['segundos'] for r in R):.1f} s")

fig = plot_anillos(R)
fig.tight_layout(); fig.savefig(OUT / "anillos.png", dpi=160); plt.show()
mostrar(tabla_anillos(R), 4)

comp = Counter((d["n"], d["composicion"]) for r in R for d in r["anillos"])
print("por composición (anillos por frame): " +
      ", ".join(f"{n} {c}: {v / len(R):.3g}" for (n, c), v in sorted(comp.items())))
fuera = np.array([(r["anillo_min"] == 0).sum() for r in R])
print(f"átomos fuera de todo anillo de <= {MAX_ANILLO}: {fuera.mean():.2f} por frame (máx. {fuera.max()})")
if len(R) > 1:
    n30 = np.array([r["tamanos"].get(30, 0) for r in R])
    # print(f"frames con todos los anillos de 30 intactos ({n30.max()}): {(n30 == n30.max()).mean():.3f} del total")

## 5. Archivos de salida

- `gr.csv`: r, g total y parciales, n_X_Y(r)
- `angulos.csv`: θ, P(θ) y P(θ)/sin θ por triplete
- `coordinacion.csv`
- `anillos.csv`: por tamaño, anillos por celda, por átomo y fracción de átomos
- `anillos_lista.csv`: átomos de cada anillo, por frame
- `atomos_anillos.extxyz`: último frame analizado con, por átomo, `coord`, `anillo_min` y `n_anillos` (para colorear en OVITO)
- figuras: `gr.png`, `gr_total.png`, `gr_cortes.png`, `angulos.png`, `anillos.png`